# 04 · Observation encoder and learned world model

**Status: UNEXECUTED FIRST DRAFT.** This notebook contains working implementations to inspect and refine when run. No trained capability or benchmark result is asserted.

**Purpose:** Encode public color grids into semantic tokens and predict action-conditioned latent transitions, reward, completion and uncertainty.

**Inputs:** Notebook 00 environment and XL configuration; notebook 01 runtime. Public grid observations only.

**Outputs:** GridEncoder, LatentWorldModel and action/mask utilities; a CPU metadata report after Run All.

**Expected runtime:** Under a minute for metadata and validation; full numerical forward pass is performed in 09. These are planning estimates; measure on this instance.

**Checkpoint directory:** `checkpoints/<run_id>/` under the project root. Definitions in this notebook are included in every source manifest.

**Exact resume:** Restart the AHR v2 XL kernel and Run All. This notebook defines components and runs bounded checks; it does not start a long training job. Training continuation is handled in 10 and 11 from the latest valid checkpoint.

The configuration remains the full XL design (`d_model=1024` by default). Any explicitly identified synthetic fixture verifies plumbing only and is not a smaller AHR training configuration.


## Load explicit notebook dependencies
Only tagged definitions are loaded; dependency execution cells are not run.

In [ ]:
from pathlib import Path
import json
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'notebooks' / '00_setup_and_hardware.ipynb').exists()), None)
if ROOT is None:
    raise RuntimeError('Open a kernel in /workspace/arc-agi-3-ahrv2 or its notebooks directory.')
_nb = json.loads((ROOT / 'notebooks' / '00_setup_and_hardware.ipynb').read_text())
for _cell in _nb['cells']:
    if _cell['cell_type'] == 'code' and 'ahr-export' in _cell.get('metadata', {}).get('tags', []):
        exec(compile(''.join(_cell['source']), '00_setup_and_hardware.ipynb', 'exec'), globals())
load_exports([1], globals())

## Grid and action validation

The 64×64 palette is discrete. ACTION6 coordinates use normalized x,y during modeling and integer pixels at the environment boundary. Masks reject empty action sets; reset is represented explicitly.

In [ ]:
import math
import copy
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F

def validate_grid_tensor(grids, grid_size=64, n_colors=16):
    grids = torch.as_tensor(grids)
    if grids.ndim == 2:
        grids = grids.unsqueeze(0)
    if grids.ndim != 3 or tuple(grids.shape[-2:]) != (grid_size, grid_size):
        raise ValueError(f'Expected [B,{grid_size},{grid_size}] grid, got {tuple(grids.shape)}')
    if grids.is_floating_point() and not torch.equal(grids, grids.round()):
        raise ValueError('Grid colors must be integers.')
    grids = grids.long()
    if grids.numel() and (bool((grids < 0).any()) or bool((grids >= n_colors).any())):
        raise ValueError('Observation has colors outside the configured palette.')
    return grids

def validate_action_mask(mask, batch_size=None, action_dim=8):
    mask = torch.as_tensor(mask, dtype=torch.bool)
    if mask.ndim == 1:
        mask = mask.unsqueeze(0)
    if mask.ndim != 2 or mask.shape[-1] != action_dim:
        raise ValueError('Action mask must have shape [B,8].')
    if batch_size is not None and mask.shape[0] != batch_size:
        raise ValueError('Action mask batch differs from context batch.')
    if not bool(mask.any(dim=-1).all()):
        raise ValueError('Each observation must expose at least one available action.')
    return mask

def mask_logits(logits, mask):
    mask = validate_action_mask(mask, logits.shape[0], logits.shape[-1]).to(logits.device)
    while mask.ndim < logits.ndim:
        mask = mask.unsqueeze(1)
    return logits.masked_fill(~mask, torch.finfo(logits.dtype).min)

def normalized_xy_to_pixels(xy, grid_size=64):
    return ((xy.clamp(-1, 1) + 1) * (grid_size - 1) / 2).round().long()

## Full XL grid encoder

Color embeddings preserve palette identity. Learned row/column positions and patch projection feed an eight-layer, width-1024 Transformer at the default configuration. Pooling returns a semantic latent; it is not an exact game-state oracle.

In [ ]:
class GridEncoder(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg['model']
        self.grid_size = int(m['grid_size'])
        self.n_colors = int(m['n_colors'])
        patch = int(m['patch_size'])
        width = int(m['d_model'])
        if self.grid_size % patch or width % int(m['n_heads']):
            raise ValueError('Grid size must divide by patch size and width by heads.')
        side = self.grid_size // patch
        color_width = max(32, width // 8)
        self.colors = nn.Embedding(self.n_colors, color_width)
        self.patch_projection = nn.Conv2d(color_width, width, kernel_size=patch, stride=patch)
        self.row_position = nn.Parameter(torch.randn(side, width) * 0.02)
        self.column_position = nn.Parameter(torch.randn(side, width) * 0.02)
        layer = nn.TransformerEncoderLayer(width, int(m['n_heads']), dim_feedforward=4*width,
                                          dropout=float(m['dropout']), activation='gelu',
                                          batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, int(m['encoder_layers']), enable_nested_tensor=False)
        self.output_norm = nn.LayerNorm(width)

    def forward(self, grids):
        grids = validate_grid_tensor(grids, self.grid_size, self.n_colors).to(self.colors.weight.device)
        colors = self.colors(grids).permute(0, 3, 1, 2)
        patches = self.patch_projection(colors).permute(0, 2, 3, 1)
        patches = patches + self.row_position[None, :, None, :] + self.column_position[None, None, :, :]
        tokens = self.output_norm(self.transformer(patches.flatten(1, 2)))
        return tokens, tokens.mean(dim=1)

## Action-conditioned dynamics and uncertainty

Three prediction heads share a dynamics trunk. Their disagreement and a learned positive variance form an uncertainty proxy that must be calibrated on held-out transitions. Neither predicted completion nor low uncertainty verifies success; the public environment does.

In [ ]:
class LatentWorldModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        m = cfg['model']
        width = int(m['d_model'])
        self.width = width
        self.action_embedding = nn.Embedding(int(m['action_dim']), width)
        self.xy_embedding = nn.Sequential(nn.Linear(2, width), nn.GELU(), nn.Linear(width, width))
        self.token_type = nn.Parameter(torch.randn(3, width) * 0.02)
        layer = nn.TransformerEncoderLayer(width, int(m['n_heads']), 4*width,
                                          float(m['dropout']), activation='gelu',
                                          batch_first=True, norm_first=True)
        self.transformer = nn.TransformerEncoder(layer, int(m['world_layers']), enable_nested_tensor=False)
        self.norm = nn.LayerNorm(width)
        self.next_heads = nn.ModuleList([nn.Linear(width, width) for _ in range(3)])
        self.reward_head = nn.Linear(width, 1)
        self.done_head = nn.Linear(width, 1)
        self.variance_head = nn.Linear(width, 1)

    def forward(self, latent, action, xy):
        action = action.long().to(latent.device)
        xy = xy.to(device=latent.device, dtype=latent.dtype)
        if latent.ndim != 2 or action.shape != latent.shape[:1] or xy.shape != (latent.shape[0], 2):
            raise ValueError('World model requires latent[B,D], action[B], xy[B,2].')
        if bool(((action < 0) | (action >= self.action_embedding.num_embeddings)).any()):
            raise ValueError('Action index outside configured action vocabulary.')
        xy = torch.where((action == 6).unsqueeze(-1), xy.clamp(-1, 1), torch.zeros_like(xy))
        tokens = torch.stack((latent, self.action_embedding(action), self.xy_embedding(xy)), dim=1)
        hidden = self.norm(self.transformer(tokens + self.token_type)[..., 0, :])
        members = torch.stack([latent + head(hidden) for head in self.next_heads], dim=1)
        prediction = members.mean(dim=1)
        disagreement = members.float().var(dim=1, unbiased=False).mean(dim=-1)
        variance = F.softplus(self.variance_head(hidden).squeeze(-1).float()) + disagreement + 1e-4
        return {'next_latent': prediction, 'reward': self.reward_head(hidden).squeeze(-1),
                'done_logit': self.done_head(hidden).squeeze(-1), 'uncertainty': variance,
                'ensemble_next_latent': members, 'disagreement': disagreement}

## Run All: bounded validation

This cell validates public input conventions and constructs the full modules on the `meta` device to count parameters without allocating training weights. No CUDA workload is launched here.

In [ ]:
cfg = load_config()
assert validate_grid_tensor(np.zeros((64, 64), dtype=np.int64)).shape == (1, 64, 64)
assert normalized_xy_to_pixels(torch.tensor([[-1.0, 1.0]])).tolist() == [[0, 63]]
with torch.device('meta'):
    encoder_metadata = GridEncoder(cfg)
    world_metadata = LatentWorldModel(cfg)
print({'encoder_parameters': sum(p.numel() for p in encoder_metadata.parameters()),
       'world_parameters': sum(p.numel() for p in world_metadata.parameters()),
       'd_model': cfg['model']['d_model'], 'numerical_forward': 'Run notebook 09'})
del encoder_metadata, world_metadata